# CounterFeint — Investigator GRPO Training (Colab Scaffold)

End-to-end training notebook for the **CounterFeint Investigator** under
the OpenEnv hackathon rules:

1. Spin up the FraudArena server in the Colab VM.
2. Train the Investigator with GRPO using the 70 % `ReactiveFraudster` /
   30 % `LLMFraudster` split (see `counterfeint/training/rollout_config.py`).
3. At the end, call `counterfeint.eval_suite.run_before_after` to produce
   the headline before/after PNG + markdown table that ships in the
   README.

**Critical design points:**
* Only the Investigator's weights move. The LLM Fraudster runs on a
  *frozen* base model (Llama-3.1-8B via Ollama) so we never co-train
  two LLMs against each other.
* All hyperparameters in `rollout_config.GRPOConfig` are placeholders;
  the final values get set onsite after a small sweep.
* Eval seeds (`counterfeint.eval_suite.EVAL_SEEDS`, 1001+) are disjoint
  from any seed used during training (training uses small ints + the
  rollout-step counter).

Run cells top-to-bottom on a Colab T4 (or any machine with the
FraudArena image + an Ollama endpoint at `OLLAMA_BASE_URL`).

## 1. Environment setup

In [ ]:
%%bash
set -e
if [ ! -d "OpenEnv-Hackathon" ]; then
  git clone https://github.com/USER/OpenEnv-Hackathon.git
fi
cd OpenEnv-Hackathon/counterfeint
pip install -q -e .
pip install -q -r requirements-dev.txt
# GRPO + transformers stack (set onsite to the version we land on):
pip install -q transformers accelerate peft trl datasets bitsandbytes

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path("OpenEnv-Hackathon").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT / "counterfeint")

# Point the LLM agents at Ollama running in the Colab background.
os.environ.setdefault("API_BASE_URL", "http://localhost:11434/v1")
os.environ.setdefault("MODEL_NAME", "llama3.1:8b-instruct")
os.environ.setdefault("COUNTERFEINT_ENV_URL", "http://localhost:8000")

print("Workspace:", os.getcwd())
print("Env:")
for k in ("API_BASE_URL", "MODEL_NAME", "COUNTERFEINT_ENV_URL"):
    print(f"  {k} = {os.environ.get(k)}")

## 2. Start the FraudArena server (background)

Runs the OpenEnv server in the Colab VM. The `[STEP]` log lines from
`inference.py` will appear here as training proceeds.

In [ ]:
import subprocess
import time

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print("Server PID:", server_proc.pid)

## 3. Configure the rollout split + GRPO hyperparameters

In [ ]:
from counterfeint.training import (
    DEFAULT_GRPO_CONFIG,
    DEFAULT_ROLLOUT_SPLIT,
    build_fraudster_factory_for_rollout,
)

config = DEFAULT_GRPO_CONFIG
print("Rollout split:", DEFAULT_ROLLOUT_SPLIT)
print("GRPO config:")
import json
print(json.dumps(config.to_dict(), indent=2))

## 4. Pre-training baseline

Run the held-out eval against the scripted Investigator so we have a
fair `before` snapshot to diff against the trained checkpoint.

In [ ]:
from counterfeint.eval_suite import run_eval
from counterfeint.scripted import ScriptedInvestigator

before_results = run_eval(
    tag="scripted_baseline",
    investigator_factory=lambda: ScriptedInvestigator(),
    # Smoke-test seeds; remove for full sweep.
    seeds={"task_1": [1001], "task_2": [2001], "task_3": [3001]},
)
for task, eps in before_results.items():
    for m in eps:
        print(f"  {task} seed={m.seed} score={m.grader_score:.3f} leaks={m.n_fraud_leaks}")

## 5. GRPO training loop

Pseudocode — the actual TRL/torchrun loop lands onsite. The key
ingredient is the per-rollout opponent draw: every batch of
`config.rollouts_per_step` rollouts pulls a Fraudster opponent from
`build_fraudster_factory_for_rollout`, which honours the 70/30 split.

In [ ]:
import random
from counterfeint.inference import run_three_agent_episode
from counterfeint.scripted import HeuristicAuditor

rng = random.Random(config.seed)
TRAINING_TASK_IDS = ("task_1", "task_2", "task_3")

# === REPLACE WITH ACTUAL GRPO LOOP ONSITE ===
# Sketch: at each step,
#   1. sample `rollouts_per_step` rollouts (one factory per rollout below).
#   2. score them with the existing grader_score + Track A reward channel.
#   3. compute group-relative advantage (GRPO).
#   4. apply PPO-style update with KL anchor on Llama-3 base.
for step in range(config.eval_every_n_steps):
    fraudster_factory = build_fraudster_factory_for_rollout(rng)
    task_id = TRAINING_TASK_IDS[step % len(TRAINING_TASK_IDS)]
    rollout = run_three_agent_episode(
        task_id,
        seed=step,  # training seeds < 1000 stay disjoint from EVAL_SEEDS
        fraudster_policy=fraudster_factory(),
        # `investigator_policy=YourTrainedLLMInvestigator(model)` here
        log=False,
    )
    # ... compute loss, backward, optimizer.step() ...

## 6. Final eval — `run_before_after`

This is the **headline cell**. It writes:
* `eval_outputs/eval_results.json`
* `eval_outputs/eval_summary.md`
* `eval_outputs/eval_plot.png`

…which the README embeds. After this cell finishes, copy the contents
of `eval_outputs/` into the repo and commit them — that's the demo
artefact the judges land on when they open the GitHub readme.

In [ ]:
from counterfeint.eval_suite import run_before_after
from counterfeint.scripted import ScriptedInvestigator

# Replace the `after_investigator_factory` lambda below with your trained
# Investigator policy once the GRPO loop above is producing a checkpoint.

summary = run_before_after(
    before_tag="scripted_baseline",
    after_tag="trained_grpo_v1",
    before_investigator_factory=lambda: ScriptedInvestigator(),
    after_investigator_factory=lambda: ScriptedInvestigator(),  # TODO: load checkpoint
    out_dir=Path("eval_outputs"),
)
print(json.dumps(summary["before"], indent=2))
print(json.dumps(summary["after"], indent=2))

In [ ]:
from IPython.display import Image, Markdown, display

display(Image(filename="eval_outputs/eval_plot.png"))
display(Markdown(Path("eval_outputs/eval_summary.md").read_text()))

## 7. Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=5)
print("Server stopped.")